# Week 4 Task 4 — Compare Approach A and Approach B Model Results

Approach A (raw files) and Approach B (platform) produce the same training dataset. Both are then processed by the **same** `w4_task2.ipynb` and `w4_task3.ipynb` notebooks, which import the shared feature pipeline and regressor from `src/data_intensive_computing/week4/`. Only the input dataset differs, selected with `WEEK4_VARIANT`.

This notebook checks whether the two runs are equivalent at three levels:

1. **Run configuration** — split profiles, feature dimension, hyperparameters, pipeline stages, and Spark settings.
2. **Fitted preprocessing** — category mappings, imputation medians, one-hot sizes, and robust-scaling statistics.
3. **Model metrics** — MAE, RMSE, and R² for the model and the Train-mean baseline.

## Prerequisites

Run from the repository root:

```bash
uv run jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 --ExecutePreprocessor.kernel_name=python3 notebooks/w4_task1.ipynb --output-dir /tmp/week4_runs
uv run jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 --ExecutePreprocessor.kernel_name=python3 notebooks/w4_task2.ipynb --output-dir /tmp/week4_runs
uv run jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 --ExecutePreprocessor.kernel_name=python3 notebooks/w4_task3.ipynb --output-dir /tmp/week4_runs
uv run jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 --ExecutePreprocessor.kernel_name=python3 notebooks/w4_task4_approach_a_raw_dataset.ipynb --output-dir /tmp/week4_runs
WEEK4_VARIANT=raw uv run jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 --ExecutePreprocessor.kernel_name=python3 notebooks/w4_task2.ipynb --output-dir /tmp/week4_runs/raw
WEEK4_VARIANT=raw uv run jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 --ExecutePreprocessor.kernel_name=python3 notebooks/w4_task3.ipynb --output-dir /tmp/week4_runs/raw
```

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from pyspark.ml import PipelineModel


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root containing pyproject.toml")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
ARTIFACTS = PROJECT_ROOT / "artifacts"
MANIFEST_PATHS = {
    "B_platform": ARTIFACTS / "week4_fare_metrics.json",
    "A_raw": ARTIFACTS / "task4_approach_a_fare_metrics.json",
}

source_package = str(PROJECT_ROOT / "src")
if source_package not in sys.path:
    sys.path.insert(0, source_package)

from data_intensive_computing.spark import build_spark_session
from data_intensive_computing.week4.features import CONTINUOUS_NUMERIC_COLS

spark = build_spark_session(
    "Week4-Task4-Compare-Approaches",
    driver_memory="4g",
    shuffle_partitions=8,
)

manifests = {name: json.loads(path.read_text(encoding="utf-8")) for name, path in MANIFEST_PATHS.items()}
for name, manifest in manifests.items():
    print(f"{name}: variant={manifest.get('variant')}, run_id={manifest['run_id']}, recorded_at={manifest['recorded_at']}")

assert manifests["B_platform"].get("variant") == "platform"
assert manifests["A_raw"].get("variant") == "raw"

:: loading settings :: url = jar:file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rexchang/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rexchang/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a76a1e41-8ccc-407f-98e4-bb7b57425d69;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in c

B_platform: variant=platform, run_id=a6d21e03-8d7b-4092-ba4e-8fb644a58aca, recorded_at=2026-10-04T08:52:42.481910+00:00
A_raw: variant=raw, run_id=869d207f-5881-40c7-a78a-ef77ae6521db, recorded_at=2026-10-04T08:59:36.959967+00:00


## 1. Run configuration

In [2]:
a, b = manifests["A_raw"], manifests["B_platform"]

split_rows = []
for split in ("train", "validation", "test"):
    a_split = a["input"]["task2_feature_dataset"]["splits"][split]
    b_split = b["input"]["task2_feature_dataset"]["splits"][split]
    split_rows.append(
        {
            "split": split,
            "rows_A": a_split["row_count"],
            "rows_B": b_split["row_count"],
            "mean_label_abs_diff": abs(a_split["mean_label"] - b_split["mean_label"]),
            "time_range_equal": (a_split["first_pickup"], a_split["last_pickup"])
            == (b_split["first_pickup"], b_split["last_pickup"]),
        }
    )
display(pd.DataFrame(split_rows))

configuration_checks = {
    "feature_dimension": (a["features"]["vector_dimension"], b["features"]["vector_dimension"]),
    "hyperparameters": (a["hyperparameters"], b["hyperparameters"]),
    "pipeline_stages": (a["pipeline_stages"], b["pipeline_stages"]),
    "spark": (
        {k: v for k, v in a["spark"].items()},
        {k: v for k, v in b["spark"].items()},
    ),
}
for name, (a_value, b_value) in configuration_checks.items():
    print(f"{name}: {'equal' if a_value == b_value else 'DIFFERENT'}")
    if a_value != b_value:
        print("  A:", a_value)
        print("  B:", b_value)

,split,rows_A,rows_B,mean_label_abs_diff,time_range_equal
0,train,5870362,5870362,3.872458e-13,True
1,validation,1726201,1726201,2.131628e-14,True
2,test,1783894,1783894,6.039613e-14,True


feature_dimension: equal
hyperparameters: equal
pipeline_stages: equal
spark: equal


## 2. Fitted preprocessing statistics

The two feature pipelines were fitted separately, each on its own Train split. Identical Train rows should give identical statistics, with one caveat: `Imputer` (median) and `RobustScaler` (quartiles) use Spark's **approximate** quantile algorithm with a default relative error of 0.001. Its answer can depend on the order in which rows arrive, so different file layouts may give slightly different medians or quartiles even for the same rows.

In [3]:
fitted = {
    "A_raw": PipelineModel.load(str(PROJECT_ROOT / "data" / "models" / "raw" / "fare_feature_pipeline")),
    "B_platform": PipelineModel.load(str(PROJECT_ROOT / "data" / "models" / "fare_feature_pipeline")),
}


def stage_by_class(model: PipelineModel, class_name: str):
    matches = [s for s in model.stages if s.__class__.__name__ == class_name]
    assert len(matches) == 1, f"Expected one {class_name}, found {len(matches)}"
    return matches[0]


a_model, b_model = fitted["A_raw"], fitted["B_platform"]

a_labels = stage_by_class(a_model, "StringIndexerModel").labelsArray
b_labels = stage_by_class(b_model, "StringIndexerModel").labelsArray
print("StringIndexer category mappings equal:", a_labels == b_labels)

a_sizes = stage_by_class(a_model, "OneHotEncoderModel").categorySizes
b_sizes = stage_by_class(b_model, "OneHotEncoderModel").categorySizes
print("OneHotEncoder category sizes equal:", a_sizes == b_sizes)

a_surrogates = stage_by_class(a_model, "ImputerModel").surrogateDF.first().asDict()
b_surrogates = stage_by_class(b_model, "ImputerModel").surrogateDF.first().asDict()
display(
    pd.DataFrame(
        {"median_A": a_surrogates, "median_B": b_surrogates}
    ).assign(abs_diff=lambda df: (df["median_A"] - df["median_B"]).abs())
)

a_scaler = stage_by_class(a_model, "RobustScalerModel")
b_scaler = stage_by_class(b_model, "RobustScalerModel")
continuous_columns = CONTINUOUS_NUMERIC_COLS
display(
    pd.DataFrame(
        {
            "feature": continuous_columns,
            "iqr_A": a_scaler.range.toArray(),
            "iqr_B": b_scaler.range.toArray(),
        }
    ).assign(abs_diff=lambda df: (df["iqr_A"] - df["iqr_B"]).abs())
)

StringIndexer category mappings equal: True
OneHotEncoder category sizes equal: True


,median_A,median_B,abs_diff
passenger_count,1.0,1.0,0.0
weather_relative_humidity_pct,56.0,56.0,0.0
weather_precipitation_mm,0.0,0.0,0.0
air_quality_pm25_mean,6.2,6.2,0.0


,feature,iqr_A,iqr_B,abs_diff
0,trip_distance,2.11,2.11,0.0
1,passenger_count_imputed,0.00,0.00,0.0
2,pickup_month_of_year,1.00,1.00,0.0
3,weather_temperature_c,5.00,5.00,0.0
4,weather_relative_humidity_pct_imputed,27.00,27.00,0.0
5,weather_precipitation_mm_imputed,0.00,0.00,0.0
6,weather_wind_speed_kmh,11.10,11.10,0.0
7,weather_pressure_hpa,10.80,10.80,0.0
8,air_quality_pm25_mean_imputed,0.00,0.00,0.0


## 3. Model metrics

In [4]:
metric_rows = []
for estimator in ("model", "train_mean_baseline"):
    for split in ("validation", "test"):
        for metric in ("mae", "rmse", "r2"):
            a_value = a["metrics"][estimator][split][metric]
            b_value = b["metrics"][estimator][split][metric]
            metric_rows.append(
                {
                    "estimator": estimator,
                    "split": split,
                    "metric": metric,
                    "A_raw": a_value,
                    "B_platform": b_value,
                    "abs_diff": abs(a_value - b_value),
                }
            )

metric_table = pd.DataFrame(metric_rows)
display(metric_table)
print("Largest absolute metric difference:", metric_table["abs_diff"].max())

,estimator,split,metric,A_raw,B_platform,abs_diff
0,model,validation,mae,6.806213,6.806221,8.047602e-06
1,model,validation,rmse,10.548764,10.548773,9.592730e-06
2,model,validation,r2,0.611811,0.611811,7.060140e-07
3,model,test,mae,6.996986,6.996995,8.963515e-06
4,model,test,rmse,10.810093,10.810104,1.061178e-05
5,model,test,r2,0.617764,0.617763,7.504481e-07
6,train_mean_baseline,validation,mae,10.898993,10.898993,1.705303e-13
7,train_mean_baseline,validation,rmse,16.937416,16.937416,1.314859e-11
8,train_mean_baseline,validation,r2,-0.000769,-0.000769,2.529532e-12
9,train_mean_baseline,test,mae,11.190630,11.190630,1.044498e-12


Largest absolute metric difference: 1.0611779066138638e-05


## Interpretation guide

| Observation | Meaning |
|---|---|
| All configuration checks equal, preprocessing identical, metric differences around `1e-12` or smaller | Equivalent runs. Remaining differences come from floating-point summation order. |
| Category mappings equal but medians or IQRs differ slightly; metric differences around `1e-4` | Equivalent data. Differences come from approximate quantiles, not from data preparation. |
| Split row counts, category mappings, or feature dimension differ | The training datasets are not equivalent. Re-run the equivalence check in `w4_task4_approach_a_raw_dataset.ipynb`. |

In [6]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F
from data_intensive_computing.week4.features import CONTINUOUS_NUMERIC_COLS
import pandas as pd

# check 1: how the scaled values actually look like
train_sample = (
    spark.read.format("delta")
    .load(str(PROJECT_ROOT / "data" / "processed" / "ml" / "fare_training_dataset"))
    .filter(F.col("split") == "train")
    .limit(200_000)
)
scaled = fitted["B_platform"].transform(train_sample).select(
    vector_to_array("continuous_features_raw").alias("raw"),
    vector_to_array("continuous_features_scaled").alias("scaled"),
)
stats = ["raw_min", "raw_max", "scaled_min", "scaled_max"]
row = scaled.agg(
    *[
        getattr(F, stat.split("_")[1])(F.col(stat.split("_")[0])[i]).alias(f"{i}_{stat}")
        for i in range(len(CONTINUOUS_NUMERIC_COLS))
        for stat in stats
    ]
).first()
display(
    pd.DataFrame(
        [
            {"feature": name, **{stat: row[f"{i}_{stat}"] for stat in stats}}
            for i, name in enumerate(CONTINUOUS_NUMERIC_COLS)
        ]
    )
)

# check 2: what are the coefficients of the fitted linear regression model
complete_model = PipelineModel.load(str(PROJECT_ROOT / "data" / "models" / "fare_prediction_pipeline"))
coefficients = complete_model.stages[-1].coefficients.toArray()
for i, name in enumerate(CONTINUOUS_NUMERIC_COLS):
    print(f"[{i}] {name:42s} coefficient = {coefficients[i]:.6f}")

,feature,raw_min,raw_max,scaled_min,scaled_max
0,trip_distance,0.0,10879.28,0.000000,5156.056872
1,passenger_count_imputed,0.0,8.00,0.000000,0.000000
2,pickup_month_of_year,1.0,1.00,1.000000,1.000000
3,weather_temperature_c,-7.2,15.00,-1.440000,3.000000
4,weather_relative_humidity_pct_imputed,23.0,97.00,0.851852,3.592593
5,weather_precipitation_mm_imputed,0.0,9.90,0.000000,0.000000
6,weather_wind_speed_kmh,0.0,57.60,0.000000,5.189189
7,weather_pressure_hpa,982.0,1034.90,90.925926,95.824074
8,air_quality_pm25_mean_imputed,1.0,26.25,0.000000,0.000000


[0] trip_distance                              coefficient = 0.000520
[1] passenger_count_imputed                    coefficient = 0.000000
[2] pickup_month_of_year                       coefficient = -0.004539
[3] weather_temperature_c                      coefficient = 0.009875
[4] weather_relative_humidity_pct_imputed      coefficient = -0.038800
[5] weather_precipitation_mm_imputed           coefficient = 0.000000
[6] weather_wind_speed_kmh                     coefficient = -0.027899
[7] weather_pressure_hpa                       coefficient = 0.039728
[8] air_quality_pm25_mean_imputed              coefficient = 0.000000
